# Import bibliotek

In [ ]:
import requests
from bs4 import BeautifulSoup
import os
import json

# Pobranie z serwisu [books.toscrape.com](books.toscrape.com) informacji o wybranych książkach

## Wysłanie żądania dostępu do zasobu (strony WWW)

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36",
    "Host": "books.toscrape.com"
}

In [ ]:
url = "https://books.toscrape.com/"
response = requests.get(url)
print(response.status_code)

200


## Parsowanie strony z informacjami o książkach

In [ ]:
page_dom = BeautifulSoup(response.text, 'html.parser')
#print(type(page_dom))
#print(page_dom)

In [ ]:
books = page_dom.select("article.product_pod")
print(type(books))
print(len(books))

<class 'bs4.element.ResultSet'>
20


### Wyszukanie informacji o jednej książce

In [ ]:
print(books[0].prettify())

<article class="product_pod">
 <div class="image_container">
  <a href="catalogue/a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   Â£51.77
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   In stock
  </p>
  <form>
   <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">
    Add to basket
   </button>
  </form>
 </div>
</article>



### Analiza informacji o pojedynczej książce

|składowa|nazwa|selektor|
|--------|-----|--------|
|nazwa|title|("h3 a")["title"]|
|cena|price|p.price_color|
|dostepność|avaliability|p.instock|
|ocena|rating|p.star-rating|

In [ ]:
i = 5
title = books[i].select_one("h3 a")["title"]
price = books[i].select_one("p.price_color").text
availability = books[i].select_one("p.instock").text.strip()
rating = books[i].select_one("p.star-rating")["class"][1]
print(f"Title: {title},\nPrice: {price},\nAvailability: {availability},\nRating: {rating}")

Title: The Requiem Red,
Price: Â£22.65,
Availability: In stock,
Rating: One


### Wybranie informacji o książkach z jednej strony

In [ ]:
all_books = []

for book in books:

    # tytuł
    title = book.select_one("h3 a")["title"]

    # cena
    price = book.select_one("p.price_color").text

    # dostępność
    availability = book.select_one("p.instock").text.strip()

    # ocena
    rating = book.select_one("p.star-rating")["class"][1]

    single_book = {
        "title": title,
        "price": price,
        "availability": availability,
        "rating": rating
    }

    all_books.append(single_book)

print(all_books[:3])
print(len(all_books))

[{'title': 'A Light in the Attic', 'price': 'Â£51.77', 'availability': 'In stock', 'rating': 'Three'}, {'title': 'Tipping the Velvet', 'price': 'Â£53.74', 'availability': 'In stock', 'rating': 'One'}, {'title': 'Soumission', 'price': 'Â£50.10', 'availability': 'In stock', 'rating': 'One'}]
20


### Wybranie informacji o książkach z 10 kolejnych stron

In [ ]:
all_books = []

for page in range(1, 10):

    url = f"https://books.toscrape.com/catalogue/page-{page}.html"
    response = requests.get(url, headers=headers)
    page_dom = BeautifulSoup(response.text, "html.parser")
    books = page_dom.select("article.product_pod")
    print(f"{url} => {response.status_code} => {len(books)}")

    for book in books:
        title = book.select_one("h3 a")["title"]
        price = book.select_one("p.price_color").text
        availability = book.select_one("p.instock").text.strip()
        rating = book.select_one("p.star-rating")["class"][1]
        single_book = {
            "title": title,
            "price": price,
            "availability": availability,
            "rating": rating
        }

        all_books.append(single_book)

print(len(all_books))

https://books.toscrape.com/catalogue/page-1.html => 200 => 20
https://books.toscrape.com/catalogue/page-2.html => 200 => 20
https://books.toscrape.com/catalogue/page-3.html => 200 => 20
https://books.toscrape.com/catalogue/page-4.html => 200 => 20
https://books.toscrape.com/catalogue/page-5.html => 200 => 20
https://books.toscrape.com/catalogue/page-6.html => 200 => 20
https://books.toscrape.com/catalogue/page-7.html => 200 => 20
https://books.toscrape.com/catalogue/page-8.html => 200 => 20
https://books.toscrape.com/catalogue/page-9.html => 200 => 20
180


# Zapis pobranych informacji

In [ ]:
if not os.path.exists("./books"):
    os.mkdir("./books")

In [ ]:
with open("./books/books_info.jason","w", encoding="UTF-8") as jf:
    json.dump(all_books, jf, indent=4, ensure_ascii=False)